In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import sqlite3
import os
drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization")
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DIR = DATA_DIR / "Raw"
DATABASE_DIR = PROJECT_ROOT / "Database"
EXPORT_DIR = DATA_DIR / "Exports"
TRAIN_FILE = RAW_DIR / "train.csv"
DB_FILE = DATABASE_DIR / "demand_forecast.db"
print("Project Root :", PROJECT_ROOT)
print("Raw Dataset  :", TRAIN_FILE)
print("Database     :", DB_FILE)
assert PROJECT_ROOT.exists(), "Project folder not found."
assert RAW_DIR.exists(), "Raw data folder not found."
assert DATABASE_DIR.exists(), "Database folder not found."
assert TRAIN_FILE.exists(), "train.csv not found."
print("\nProject environment verified successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project Root : /content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization
Raw Dataset  : /content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Data/Raw/train.csv
Database     : /content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db

Project environment verified successfully.


In [28]:

import shutil

print("CONNECTING TO SQLITE DATABASE")

DRIVE_DB_FILE = DB_FILE
LOCAL_DB_FILE = Path("/content/demand_forecast.db")

if DRIVE_DB_FILE.exists():
    shutil.copy2(DRIVE_DB_FILE, LOCAL_DB_FILE)
    print("Existing database copied to local storage.")
else:
    print("Creating new local database.")

try:
    conn.close()
except:
    pass

conn = sqlite3.connect(LOCAL_DB_FILE)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

def checkpoint_to_drive():
    conn.commit()
    shutil.copy2(LOCAL_DB_FILE, DRIVE_DB_FILE)
    print("Database copied back to Google Drive.")

print("\nWorking Database")
print(LOCAL_DB_FILE)

print("\nConnection Status")
print("SQLite database connected successfully.")

CONNECTING TO SQLITE DATABASE
Existing database copied to local storage.

Working Database
/content/demand_forecast.db

Connection Status
SQLite database connected successfully.


In [3]:
print("LOADING VALIDATED SALES DATASET")
df = pd.read_csv(TRAIN_FILE)
df["date"] = pd.to_datetime(df["date"])
print("\nDataset Information")
print(f"Rows              : {len(df):,}")
print(f"Columns           : {df.shape[1]}")
print(f"Stores            : {df['store'].nunique()}")
print(f"Items             : {df['item'].nunique()}")
print(f"Date Range        : {df['date'].min().date()} to {df['date'].max().date()}")
print("\nValidation")
assert len(df) == 913000, "Unexpected number of rows."
assert df["store"].nunique() == 10, "Unexpected number of stores."
assert df["item"].nunique() == 50, "Unexpected number of items."
print("Dataset successfully loaded and validated.")
print("\n")
print("DATA READY FOR SQL PIPELINE")

LOADING VALIDATED SALES DATASET

Dataset Information
Rows              : 913,000
Columns           : 4
Stores            : 10
Items             : 50
Date Range        : 2013-01-01 to 2017-12-31

Validation
Dataset successfully loaded and validated.


DATA READY FOR SQL PIPELINE


In [29]:
print("Checking local database...")

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

print(tables)

Checking local database...
               name
0          dim_date
1          dim_item
2         dim_store
3  fact_daily_sales
4     stg_sales_raw


In [30]:
rows = pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM fact_daily_sales
""", conn)

print(rows)

   total_rows
0           5


In [31]:
cursor.execute("DELETE FROM fact_daily_sales")
conn.commit()

print("fact_daily_sales cleared successfully.")

fact_daily_sales cleared successfully.


In [32]:
import time

start = time.time()

cursor.execute("""
INSERT INTO fact_daily_sales (
    date_key,
    store_key,
    item_key,
    sales_qty
)
SELECT
    CAST(strftime('%Y%m%d', date) AS INTEGER),
    store,
    item,
    sales
FROM stg_sales_raw
""")

conn.commit()

elapsed = time.time() - start

rows = pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM fact_daily_sales
""", conn).iloc[0, 0]

print(f"Rows Loaded : {rows:,}")
print(f"Time Taken  : {elapsed:.2f} seconds")

assert rows == 913000

print("\nfact_daily_sales populated successfully.")

Rows Loaded : 913,000
Time Taken  : 7.30 seconds

fact_daily_sales populated successfully.


In [33]:
checkpoint_to_drive()

Database copied back to Google Drive.


In [34]:
import sqlite3
import pandas as pd

drive_conn = sqlite3.connect(DRIVE_DB_FILE)

rows = pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM fact_daily_sales
""", drive_conn)

print(rows)

drive_conn.close()

   total_rows
0      913000


In [4]:

print("CREATING STAGING TABLE")
cursor.execute("DROP TABLE IF EXISTS stg_sales_raw")
cursor.execute("""
CREATE TABLE stg_sales_raw (
    date TEXT NOT NULL,
    store INTEGER NOT NULL,
    item INTEGER NOT NULL,
    sales INTEGER NOT NULL
)
""")
conn.commit()
print("\nTable Name")
print("stg_sales_raw")
print("\nColumns")
print("date   : TEXT")
print("store  : INTEGER")
print("item   : INTEGER")
print("sales  : INTEGER")
print("\nStatus")
print("Staging table created successfully.")
print("STAGING TABLE READY")

CREATING STAGING TABLE

Table Name
stg_sales_raw

Columns
date   : TEXT
store  : INTEGER
item   : INTEGER
sales  : INTEGER

Status
Staging table created successfully.
STAGING TABLE READY


In [5]:
df.to_sql(
    "stg_sales_raw",
    conn,
    if_exists="replace",
    index=False
)

cursor.execute("SELECT COUNT(*) FROM stg_sales_raw")
staging_rows = cursor.fetchone()[0]

assert staging_rows == len(df)

print(f"Rows loaded into stg_sales_raw : {staging_rows:,}")

Rows loaded into stg_sales_raw : 913,000


In [6]:
print("Database File:")
print(DB_FILE)

print("\nConnection Object:")
print(conn)

cursor.execute("PRAGMA database_list;")
print("\nConnected Database:")
print(cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM stg_sales_raw")
print("\nRows in stg_sales_raw:")
print(cursor.fetchone()[0])

print("\nRows in DataFrame:")
print(len(df))

Database File:
/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db

Connection Object:

Connected Database:
[(0, 'main', '/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db')]

Rows in stg_sales_raw:
913000

Rows in DataFrame:
913000


In [7]:
cursor.execute("SELECT COUNT(*) FROM stg_sales_raw")
staging_rows = cursor.fetchone()[0]
dataframe_rows = len(df)
print(f"Rows in DataFrame     : {dataframe_rows:,}")
print(f"Rows in stg_sales_raw : {staging_rows:,}")
assert staging_rows == dataframe_rows
print("Load reconciliation completed successfully.")

Rows in DataFrame     : 913,000
Rows in stg_sales_raw : 913,000
Load reconciliation completed successfully.


In [8]:
cursor.execute("DROP TABLE IF EXISTS dim_date")

cursor.execute("""
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER,
    quarter INTEGER,
    month INTEGER,
    month_name TEXT,
    week INTEGER,
    day INTEGER,
    day_name TEXT,
    day_of_year INTEGER,
    is_weekend INTEGER,
    is_holiday INTEGER,
    holiday_name TEXT
)
""")

conn.commit()

print("Table created : dim_date")

Table created : dim_date


In [9]:
import holidays

date_range = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    freq="D"
)

us_holidays = holidays.US(years=range(
    date_range.min().year,
    date_range.max().year + 1
))

dim_date = pd.DataFrame({
    "full_date": date_range
})

dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["week"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["day_of_year"] = dim_date["full_date"].dt.dayofyear
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek.isin([5, 6]).astype(int)

dim_date["holiday_name"] = dim_date["full_date"].apply(
    lambda x: us_holidays.get(x)
)

dim_date["is_holiday"] = dim_date["holiday_name"].notna().astype(int)

dim_date["full_date"] = dim_date["full_date"].dt.strftime("%Y-%m-%d")

dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week",
        "day",
        "day_name",
        "day_of_year",
        "is_weekend",
        "is_holiday",
        "holiday_name"
    ]
]

dim_date.to_sql(
    "dim_date",
    conn,
    if_exists="append",
    index=False
)

cursor.execute("SELECT COUNT(*) FROM dim_date")
rows = cursor.fetchone()[0]

print(f"Rows inserted into dim_date : {rows:,}")

Rows inserted into dim_date : 1,826


In [10]:
rows = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM dim_date",
    conn
).iloc[0, 0]

date_range = pd.read_sql_query(
    """
    SELECT
        MIN(full_date) AS start_date,
        MAX(full_date) AS end_date
    FROM dim_date
    """,
    conn
)

holiday_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS holidays
    FROM dim_date
    WHERE is_holiday = 1
    """,
    conn
).iloc[0, 0]

weekend_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS weekends
    FROM dim_date
    WHERE is_weekend = 1
    """,
    conn
).iloc[0, 0]

assert rows == 1826, "dim_date should contain 1,826 rows."

print(f"Rows            : {rows:,}")
print(f"Date Range      : {date_range.iloc[0,0]} to {date_range.iloc[0,1]}")
print(f"Holiday Records : {holiday_count}")
print(f"Weekend Records : {weekend_count}")

print("dim_date validation completed successfully.")

Rows            : 1,826
Date Range      : 2013-01-01 to 2017-12-31
Holiday Records : 54
Weekend Records : 522
dim_date validation completed successfully.


In [11]:
cursor.execute("DROP TABLE IF EXISTS dim_store")

cursor.execute("""
CREATE TABLE dim_store (
    store_key INTEGER PRIMARY KEY,
    store_id INTEGER,
    store_name TEXT
)
""")

dim_store = pd.DataFrame({
    "store_key": range(1, 11),
    "store_id": range(1, 11),
    "store_name": [f"Store {i}" for i in range(1, 11)]
})

dim_store.to_sql(
    "dim_store",
    conn,
    if_exists="append",
    index=False
)

rows = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM dim_store",
    conn
).iloc[0, 0]

assert rows == 10

print(f"Rows inserted into dim_store : {rows}")
print("dim_store created successfully.")

Rows inserted into dim_store : 10
dim_store created successfully.


In [12]:
cursor.execute("DROP TABLE IF EXISTS dim_item")

cursor.execute("""
CREATE TABLE dim_item (
    item_key INTEGER PRIMARY KEY,
    item_id INTEGER,
    total_sales INTEGER,
    total_revenue REAL,
    volume_rank INTEGER,
    price_band TEXT,
    unit_price REAL,
    abc_class TEXT
)
""")

conn.commit()

print("Table created : dim_item")

Table created : dim_item


In [14]:
import numpy as np

cursor.execute("DELETE FROM dim_item")
conn.commit()

item_volume = pd.read_sql_query("""
SELECT
    item,
    SUM(sales) AS total_sales,
    RANK() OVER (
        ORDER BY SUM(sales) DESC
    ) AS volume_rank
FROM stg_sales_raw
GROUP BY item
ORDER BY volume_rank
""", conn)

prices = np.logspace(
    np.log10(40),
    np.log10(2500),
    len(item_volume)
)

item_volume["unit_price"] = np.round(prices, 2)

item_volume["price_band"] = pd.cut(
    item_volume["unit_price"],
    bins=[0,150,600,3000],
    labels=["Low","Mid","High"]
)

item_volume["total_revenue"] = (
    item_volume["total_sales"] *
    item_volume["unit_price"]
)

item_volume = item_volume.sort_values(
    "total_revenue",
    ascending=False
)

item_volume["cum_pct"] = (
    item_volume["total_revenue"].cumsum()
    /
    item_volume["total_revenue"].sum()
)

item_volume["abc_class"] = np.where(
    item_volume["cum_pct"] <= 0.70,
    "A",
    np.where(
        item_volume["cum_pct"] <= 0.90,
        "B",
        "C"
    )
)

print(item_volume.head())

    item  total_sales  volume_rank  unit_price price_band  total_revenue  \
48     1       401384           49     2297.68       High   9.222520e+08   
47    41       401759           48     2111.73       High   8.484065e+08   
49     5       335230           50     2500.00       High   8.380750e+08   
46    47       401781           47     1940.83       High   7.797886e+08   
45     4       401907           46     1783.76       High   7.169056e+08   

     cum_pct abc_class  
48  0.051625         A  
47  0.099115         A  
49  0.146028         A  
46  0.189678         A  
45  0.229808         A  


In [15]:
rows = [
    (
        int(r.item),
        int(r.item),
        int(r.total_sales),
        float(r.total_revenue),
        int(r.volume_rank),
        str(r.price_band),
        float(r.unit_price),
        str(r.abc_class)
    )
    for r in item_volume.itertuples()
]

cursor.execute("DELETE FROM dim_item")

cursor.executemany("""
INSERT INTO dim_item
(
    item_key,
    item_id,
    total_sales,
    total_revenue,
    volume_rank,
    price_band,
    unit_price,
    abc_class
)
VALUES (?,?,?,?,?,?,?,?)
""", rows)

conn.commit()

print("dim_item loaded successfully.")

dim_item loaded successfully.


In [16]:
rows = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM dim_item",
    conn
).iloc[0,0]

nulls = pd.read_sql_query("""
SELECT COUNT(*) AS total_nulls
FROM dim_item
WHERE total_sales IS NULL
OR total_revenue IS NULL
OR volume_rank IS NULL
OR price_band IS NULL
OR unit_price IS NULL
OR abc_class IS NULL
""", conn).iloc[0,0]

abc = pd.read_sql_query("""
SELECT
    abc_class,
    COUNT(*) AS items
FROM dim_item
GROUP BY abc_class
ORDER BY abc_class
""", conn)

print(f"Rows Loaded : {rows}")
print(f"Null Values : {nulls}")
print()
print("ABC Distribution")
print(abc)

assert rows == 50
assert nulls == 0

print("\ndim_item validation completed successfully.")

Rows Loaded : 50
Null Values : 0

ABC Distribution
  abc_class  items
0         A     20
1         B     13
2         C     17

dim_item validation completed successfully.


In [17]:
cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("DROP TABLE IF EXISTS fact_daily_sales")

cursor.execute("""
CREATE TABLE fact_daily_sales (
    date_key INTEGER NOT NULL,
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    sales_qty INTEGER NOT NULL,
    PRIMARY KEY (
        date_key,
        store_key,
        item_key
    ),
    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key),
    FOREIGN KEY (store_key)
        REFERENCES dim_store(store_key),
    FOREIGN KEY (item_key)
        REFERENCES dim_item(item_key)
)
""")

conn.commit()

schema = pd.read_sql_query(
    "PRAGMA table_info(fact_daily_sales)",
    conn
)

print(schema[["name", "type", "notnull", "pk"]])

        name     type  notnull  pk
0   date_key  INTEGER        1   1
1  store_key  INTEGER        1   2
2   item_key  INTEGER        1   3
3  sales_qty  INTEGER        1   0


In [35]:
import time

cursor.execute("DELETE FROM fact_daily_sales")
conn.commit()

start = time.time()

cursor.execute("""
INSERT INTO fact_daily_sales (
    date_key,
    store_key,
    item_key,
    sales_qty
)
SELECT
    CAST(strftime('%Y%m%d', date) AS INTEGER),
    store,
    item,
    sales
FROM stg_sales_raw
""")

conn.commit()

elapsed = time.time() - start

rows = pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM fact_daily_sales
""", conn).iloc[0, 0]

print(f"Rows Loaded : {rows:,}")
print(f"Time Taken  : {elapsed:.2f} seconds")

assert rows == 913000

print("\nfact_daily_sales populated successfully.")

Rows Loaded : 913,000
Time Taken  : 5.24 seconds

fact_daily_sales populated successfully.


In [42]:

orphan_dates = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_daily_sales f
LEFT JOIN dim_date d
ON f.date_key = d.date_key
WHERE d.date_key IS NULL
""", conn).iloc[0, 0]

orphan_stores = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_daily_sales f
LEFT JOIN dim_store s
ON f.store_key = s.store_key
WHERE s.store_key IS NULL
""", conn).iloc[0, 0]

orphan_items = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_daily_sales f
LEFT JOIN dim_item i
ON f.item_key = i.item_key
WHERE i.item_key IS NULL
""", conn).iloc[0, 0]

print("=== REFERENTIAL INTEGRITY ===")
print(f"Orphaned date_key rows : {orphan_dates}")
print(f"Orphaned store_key rows: {orphan_stores}")
print(f"Orphaned item_key rows : {orphan_items}")
staging_total = pd.read_sql_query("""
SELECT SUM(sales) AS total
FROM stg_sales_raw
""", conn).iloc[0, 0]

fact_total = pd.read_sql_query("""
SELECT SUM(sales_qty) AS total
FROM fact_daily_sales
""", conn).iloc[0, 0]

rows = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_daily_sales
""", conn).iloc[0, 0]

print("\n=== VALUE RECONCILIATION ===")
print(f"Total units in stg_sales_raw    : {staging_total:,}")
print(f"Total units in fact_daily_sales : {fact_total:,}")
print(f"Rows Loaded                     : {rows:,}")

assert orphan_dates == 0
assert orphan_stores == 0
assert orphan_items == 0
assert staging_total == fact_total
assert rows == 913000

print("\nfact_daily_sales validation completed successfully.")

=== REFERENTIAL INTEGRITY ===
Orphaned date_key rows : 0
Orphaned store_key rows: 0
Orphaned item_key rows : 0

=== VALUE RECONCILIATION ===
Total units in stg_sales_raw    : 47,704,512
Total units in fact_daily_sales : 47,704,512
Rows Loaded                     : 913,000

fact_daily_sales validation completed successfully.


In [43]:
cursor.execute("DROP TABLE IF EXISTS dim_model")

cursor.execute("""
CREATE TABLE dim_model (
    model_key INTEGER PRIMARY KEY,
    model_name TEXT NOT NULL,
    model_type TEXT NOT NULL
)
""")

model_rows = [
    (1, "Naive", "Baseline"),
    (2, "SARIMA", "Statistical"),
    (3, "Holt-Winters", "Statistical"),
    (4, "XGBoost", "ML")
]

cursor.executemany("""
INSERT INTO dim_model (
    model_key,
    model_name,
    model_type
)
VALUES (?, ?, ?)
""", model_rows)

conn.commit()

result = pd.read_sql_query("""
SELECT *
FROM dim_model
ORDER BY model_key
""", conn)

print(result)

assert len(result) == 4

print("\ndim_model created and validated.")

   model_key    model_name   model_type
0          1         Naive     Baseline
1          2        SARIMA  Statistical
2          3  Holt-Winters  Statistical
3          4       XGBoost           ML

dim_model created and validated.


In [44]:
cursor.execute("DROP TABLE IF EXISTS fact_forecast_accuracy")

cursor.execute("""
CREATE TABLE fact_forecast_accuracy (
    date_key INTEGER NOT NULL,
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    model_key INTEGER NOT NULL,
    actual_qty INTEGER NOT NULL,
    forecast_qty REAL,
    abs_error REAL,
    abs_pct_error REAL,
    squared_error REAL,
    PRIMARY KEY (
        date_key,
        store_key,
        item_key,
        model_key
    ),
    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key),
    FOREIGN KEY (store_key)
        REFERENCES dim_store(store_key),
    FOREIGN KEY (item_key)
        REFERENCES dim_item(item_key),
    FOREIGN KEY (model_key)
        REFERENCES dim_model(model_key)
)
""")

conn.commit()

schema = pd.read_sql_query("""
PRAGMA table_info(fact_forecast_accuracy)
""", conn)

print(schema[["name", "type", "notnull", "pk"]])

            name     type  notnull  pk
0       date_key  INTEGER        1   1
1      store_key  INTEGER        1   2
2       item_key  INTEGER        1   3
3      model_key  INTEGER        1   4
4     actual_qty  INTEGER        1   0
5   forecast_qty     REAL        0   0
6      abs_error     REAL        0   0
7  abs_pct_error     REAL        0   0
8  squared_error     REAL        0   0


In [45]:
# Clear any prior naive rows so this cell is safely re-runnable
cursor.execute("DELETE FROM fact_forecast_accuracy WHERE model_key = 1")
conn.commit()

cursor.execute("""
INSERT INTO fact_forecast_accuracy (
    date_key,
    store_key,
    item_key,
    model_key,
    actual_qty,
    forecast_qty,
    abs_error,
    abs_pct_error,
    squared_error
)
SELECT
    date_key,
    store_key,
    item_key,
    1 AS model_key,
    actual_qty,
    forecast_qty,
    ABS(actual_qty - forecast_qty) AS abs_error,
    CASE
        WHEN actual_qty > 0
        THEN ABS(actual_qty - forecast_qty) * 1.0 / actual_qty
        ELSE NULL
    END AS abs_pct_error,
    (actual_qty - forecast_qty) *
    (actual_qty - forecast_qty) AS squared_error
FROM (
    SELECT
        date_key,
        store_key,
        item_key,
        sales_qty AS actual_qty,
        LAG(sales_qty, 7) OVER (
            PARTITION BY store_key, item_key
            ORDER BY date_key
        ) AS forecast_qty
    FROM fact_daily_sales
) lagged
WHERE date_key BETWEEN 20171001 AND 20171231
  AND forecast_qty IS NOT NULL
""")

conn.commit()

n_inserted = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_forecast_accuracy
WHERE model_key = 1
""", conn).iloc[0, 0]

print(f"Naive baseline rows inserted: {n_inserted}")
assert n_inserted == 46000

print("\nNaive baseline inserted successfully.")

Naive baseline rows inserted: 46000

Naive baseline inserted successfully.


In [46]:
sku_mape = pd.read_sql_query("""
SELECT
    store_key,
    item_key,
    AVG(abs_pct_error) AS sku_mape
FROM fact_forecast_accuracy
WHERE model_key = 1
  AND abs_pct_error IS NOT NULL
GROUP BY store_key, item_key
""", conn)

n_skus = len(sku_mape)
headline_mape = sku_mape["sku_mape"].mean() * 100

print(f"SKUs with a computed MAPE: {n_skus} (expected 500)")
print(
    f"Per-SKU MAPE -- min: {sku_mape['sku_mape'].min()*100:.1f}%, "
    f"median: {sku_mape['sku_mape'].median()*100:.1f}%, "
    f"max: {sku_mape['sku_mape'].max()*100:.1f}%"
)

print(f"\n*** HEADLINE NAIVE BASELINE MAPE: {headline_mape:.1f}% ***")
print("(Previously validated result from the full pipeline run: 19.9%)")

n_excluded = pd.read_sql_query("""
SELECT COUNT(*) AS n
FROM fact_forecast_accuracy
WHERE model_key = 1
  AND abs_pct_error IS NULL
""", conn).iloc[0, 0]

print(f"\nRows excluded from MAPE (zero-actual days): {n_excluded}")

assert n_skus == 500
assert abs(headline_mape - 19.9) < 0.5

print("\nNaive baseline validated: matches the previously confirmed 19.9% MAPE.")

SKUs with a computed MAPE: 500 (expected 500)
Per-SKU MAPE -- min: 10.9%, median: 18.8%, max: 40.6%

*** HEADLINE NAIVE BASELINE MAPE: 19.9% ***
(Previously validated result from the full pipeline run: 19.9%)

Rows excluded from MAPE (zero-actual days): 0

Naive baseline validated: matches the previously confirmed 19.9% MAPE.


In [48]:
checkpoint_to_drive()

drive_conn = sqlite3.connect(DB_FILE)

tables_check = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM dim_model) AS dim_model_rows,
    (SELECT COUNT(*) FROM fact_forecast_accuracy) AS fact_forecast_accuracy_rows
""", drive_conn)

drive_conn.close()

print(tables_check)

assert tables_check["dim_model_rows"][0] == 4
assert tables_check["fact_forecast_accuracy_rows"][0] == 46000

print("\nDrive copy confirmed up to date.")

Database copied back to Google Drive.
   dim_model_rows  fact_forecast_accuracy_rows
0               4                        46000

Drive copy confirmed up to date.


In [49]:
print("=" * 60)
print("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
print("=" * 60)
print("SQLite Star Schema        : Completed")
print("SQL ETL Pipeline          : Completed")
print("Naive Baseline            : Completed")
print("Naive Baseline MAPE       : 19.9%")
print("fact_daily_sales          : 913,000 rows")
print("fact_forecast_accuracy    : 46,000 rows")
print("Database Checkpoint       : Completed")
print("=" * 60)
print("Ready for Notebook 03")

NOTEBOOK 02 COMPLETED SUCCESSFULLY
SQLite Star Schema        : Completed
SQL ETL Pipeline          : Completed
Naive Baseline            : Completed
Naive Baseline MAPE       : 19.9%
fact_daily_sales          : 913,000 rows
fact_forecast_accuracy    : 46,000 rows
Database Checkpoint       : Completed
Ready for Notebook 03
